In [116]:
import numpy as np
import matplotlib.pyplot as plt

# Use LaTeX fonts for better readability
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Palatino"],
        "axes.labelsize": 12,
        "axes.titlesize": 14,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    }
)

# Define custom colors
colors = {
    "inter_token_latency": "#1f77b4",  # Blue
    "expert_load_time": "#ff7f0e",  # Orange
    "base_tokens_per_sec": "#2ca02c",  # Green
    "tokens_with_experts_amortized": "#d62728",  # Red
    "tokens_with_experts_no_amortization": "#9467bd",  # Purple
}

# Model parameters
seq_len = 4096
hidden_dim = 3072
intermediate_dim = 12288
num_heads = 24
num_layers = 30
num_kv_heads = 2
vocab_size = 32000

# Bandwidths (GB/s converted to bytes/s)
flash_bandwidth = 4e9  # 4GB/s
lpddr_bandwidth = 50e9  # 50GB/s

# Base model parameter calculations
transformer_block_params = (
    (hidden_dim * intermediate_dim) * 2  # MLP weights
    + (hidden_dim * hidden_dim) * 2  # Attention projection weights
    + (hidden_dim * num_heads)  # Query weights
    + (hidden_dim * num_kv_heads * 2)  # Key and value weights
) * num_layers

bytes_per_param = 0.5

embedding_params = vocab_size * hidden_dim
base_model_params = transformer_block_params + embedding_params
base_model_size_bytes = (
    base_model_params * bytes_per_param
)  # Convert params to bytes (4-bit precision)

# Expert calculations
expert_rank = 128
expert_params_per_layer = (
    hidden_dim * expert_rank * 2
)  # Weight matrices for MLP per expert

k_values = np.arange(1, 65)  # Range of k values

# Compute expert parameters across all layers
total_expert_params = k_values * expert_params_per_layer * num_layers
total_expert_size_bytes = total_expert_params * bytes_per_param

# Compute total model size including experts
total_model_sizes = base_model_size_bytes + total_expert_size_bytes

# Compute load times
base_model_load_time = base_model_size_bytes / lpddr_bandwidth
model_load_times_with_experts = total_model_sizes / lpddr_bandwidth
expert_load_times = total_expert_size_bytes / flash_bandwidth

# Compute tokens per second
base_tokens_per_second = 1 / base_model_load_time
tokens_per_second_amortized = 1 / (
    model_load_times_with_experts + (expert_load_times / 64)
)
tokens_per_second_no_amortization = 1 / (
    model_load_times_with_experts + expert_load_times
)

# Filter for total_expert_params <= 0.6B
mask = total_expert_params <= 0.5e9
filtered_total_expert_params = total_expert_params[mask]
filtered_model_load_times_with_experts = model_load_times_with_experts[mask]
filtered_expert_load_times = expert_load_times[mask]
filtered_tokens_per_second_amortized = tokens_per_second_amortized[mask]
filtered_tokens_per_second_no_amortization = tokens_per_second_no_amortization[mask]

# Plot load times
plt.figure(figsize=(4, 3), dpi=300)
plt.plot(
    filtered_total_expert_params,
    filtered_model_load_times_with_experts,
    label="Peak ITL (LPDDR5 @ 50GB/s)",
    linestyle="--",
    color=colors["inter_token_latency"],
)
plt.plot(
    filtered_total_expert_params,
    filtered_expert_load_times,
    label="Expert Load Time (Flash @ 4GB/s)",
    marker="o",
    color=colors["expert_load_time"],
)

# Save load times plot as a PDF
plt.xlabel("Active Expert Parameters")
plt.ylabel("Theoretical Time (s)")
plt.ylim(0, 0.09)
plt.title("Expert v.s. Model Loading Time", fontsize=12)
plt.legend(loc="upper left")
plt.grid(True)
plt.tight_layout()
plt.savefig("output/model_and_expert_load_times.pdf", format="pdf")
plt.close()

# Save tokens per second plot as a PDF
plt.figure(figsize=(4, 3), dpi=300)
plt.plot(
    filtered_total_expert_params,
    [base_tokens_per_second] * len(filtered_total_expert_params),
    label="Base Model",
    linestyle="--",
    color=colors["base_tokens_per_sec"],
)
plt.plot(
    filtered_total_expert_params,
    filtered_tokens_per_second_amortized,
    label="SLEs (Amortized)",
    marker="o",
    color=colors["tokens_with_experts_amortized"],
)
plt.plot(
    filtered_total_expert_params,
    filtered_tokens_per_second_no_amortization,
    label="MoE",
    marker="s",
    color=colors["tokens_with_experts_no_amortization"],
)


plt.xlabel("Active Expert Parameters")
plt.ylabel("Tokens/s")
plt.ylim(0)
plt.title("MoE v.s. SLE Inter-Token Latency", fontsize=12)
plt.legend(loc="lower left")
plt.grid(True)
plt.tight_layout()

plt.savefig("output/tokens_per_second_vs_expert_params.pdf", format="pdf")
plt.close()


# Scaling figures

1. LR v.s. completion loss for MLP baseline, flop equiv mlp, full ft, routed_mlp, others?
2. load balancing loss + lflb weights v.s. loss

In [118]:
from functools import partial
import pandas as pd
import seaborn as sns
from pprint import pprint

df = pd.read_csv("wandb_output.csv")
all_tags = set(a.strip() for b in df["Tags"].unique() for a in b.split(","))
#pprint(all_tags)
compare_tags = ["mlp_baseline", "mlp_flopeq_baseline", "full_ft_noanneal", "routed_mlp"]
#pprint(df.columns.tolist())


def has_tag(row, tag):
    tags = row["Tags"].split(",")
    return tag.strip() in tags


def to_target_tag(compare_tags, row):
    tags = row["Tags"].split(",")
    for tag in tags:
        if tag.strip() in compare_tags:
            return tag.strip()
    return "other"


routed_mlp = df[df.apply(lambda x: has_tag(x, "routed_mlp"), axis=1)]

df_lr_compare = df.copy()
df_lr_compare["target_tag"] = df.apply(partial(to_target_tag, compare_tags), axis=1)
df_lr_compare = df_lr_compare[df_lr_compare["target_tag"] != "other"]
plt.figure(figsize=(4, 3), dpi=300)
sns.lineplot(
    data=df_lr_compare,
    x="optimizer.learning_rate",
    y="eval/completion_loss (Min)",
    hue="target_tag",
    marker="o",
)
plt.xscale("log")
plt.tight_layout()
plt.savefig("output/lr_compare.pdf", format="pdf")
plt.close()

routed_mlp_baseline = routed_mlp["eval/completion_loss (Min)"].min()
df_lbl = df.copy()
compare_tags = ["routed_mlp_lblb"]
df_lbl["target_tag"] = df.apply(partial(to_target_tag, compare_tags), axis=1)
df_lbl = df_lbl[df_lbl["target_tag"].isin(compare_tags)]
plt.figure(figsize=(4, 3), dpi=300)
sns.lineplot(
    data=df_lbl,
    x="lb_loss_weight",
    y="eval/completion_loss (Min)",
    hue="target_tag",
    marker="o",
)
plt.axhline(routed_mlp_baseline, color="black", linestyle="--")
plt.xscale("log")
plt.tight_layout()
plt.savefig("output/lb_loss_weight.pdf", format="pdf")
plt.close()

df_lflb = df.copy()
compare_tags = ["routed_mlp_lflb"]
df_lflb["target_tag"] = df.apply(partial(to_target_tag, compare_tags), axis=1)
df_lflb = df_lflb[df_lflb["target_tag"] != "other"]
df_lflb["target_tag_lb"] = df_lflb["target_tag"] + df_lflb["lb_loss_weight"].astype(str)
plt.figure(figsize=(4, 3), dpi=300)
sns.lineplot(
    data=df_lflb,
    x="model.expert_bias_update_rate",
    y="eval/completion_loss (Min)",
    hue="target_tag_lb",
    marker="o",
)
plt.axhline(routed_mlp_baseline, color="black", linestyle="--")
plt.xscale("log")
plt.tight_layout()
plt.savefig("output/expert_bias_update_rate.pdf", format="pdf")
plt.close()

# "routed_mlp_zloss",
df_zloss = df.copy()
compare_tags = ["routed_mlp_zloss"]
df_zloss["target_tag"] = df.apply(partial(to_target_tag, compare_tags), axis=1)
df_zloss = df_zloss[df_zloss["target_tag"] != "other"]
plt.figure(figsize=(4, 3), dpi=300)
sns.lineplot(
    data=df_zloss,
    x="router_z_loss_weight",
    y="eval/completion_loss (Min)",
    hue="target_tag",
    marker="o",
)
plt.axhline(routed_mlp_baseline, color="black", linestyle="--")
plt.xscale("log")
plt.tight_layout()
plt.savefig("output/router_z_loss_weight.pdf", format="pdf")
plt.close()

### Scaling figures